In [2]:
import gzip
import json
import random

# Configuration
input_file = ("ratings/Health_and_Household.csv.gz")
output_file = "Health_and_Household_10_percent.csv"
sample_rate = 0.1  # 10%

def sample_dataset(input_path, output_path, rate):
    count = 0
    sampled_count = 0
    
    # Opening as 'rt' (read text) handles the decompression on-the-fly
    with gzip.open(input_path, 'rt', encoding='utf-8') as f_in:
        with open(output_path, 'w', encoding='utf-8') as f_out:
            for line in f_in:
                count += 1
                # Randomly decide to keep this line
                if random.random() < rate:
                    f_out.write(line)
                    sampled_count += 1
                
                # Print progress every 100k lines
                if count % 100000 == 0:
                    print(f"Processed {count} lines... Saved {sampled_count}")

    print(f"Finished! Total: {count} | Sampled: {sampled_count}")

sample_dataset(input_file, output_file, sample_rate)

Processed 100000 lines... Saved 9984
Processed 200000 lines... Saved 20038
Processed 300000 lines... Saved 30101
Processed 400000 lines... Saved 40050
Processed 500000 lines... Saved 50062
Processed 600000 lines... Saved 59929
Processed 700000 lines... Saved 69973
Processed 800000 lines... Saved 79965
Processed 900000 lines... Saved 90175
Processed 1000000 lines... Saved 100159
Processed 1100000 lines... Saved 110165
Processed 1200000 lines... Saved 120123
Processed 1300000 lines... Saved 130175
Processed 1400000 lines... Saved 140213
Processed 1500000 lines... Saved 150416
Processed 1600000 lines... Saved 160530
Processed 1700000 lines... Saved 170442
Processed 1800000 lines... Saved 180445
Processed 1900000 lines... Saved 190585
Processed 2000000 lines... Saved 200739
Processed 2100000 lines... Saved 210791
Processed 2200000 lines... Saved 220864
Processed 2300000 lines... Saved 230900
Processed 2400000 lines... Saved 240682
Processed 2500000 lines... Saved 250872
Processed 2600000 l

Read Meta

In [3]:
import pandas as pd

# Load the filter
sample_df = pd.read_csv("samples/Health_and_Household_10_percent.csv", header=None, names=['u', 'parent_asin', 'r', 't'])
valid_asins = set(sample_df['parent_asin'].unique())

# Use the built-in Pandas JSON parser
# chunksize keeps memory low; lines=True handles the format
print("Reading with built-in JSON parser...")
chunks = pd.read_json("meta/meta_Health_and_Household.jsonl.gz", lines=True, chunksize=1000, compression='gzip')

full_meta = []
for chunk in chunks:
    # Filter each chunk against your ASIN list
    filtered = chunk[chunk['parent_asin'].isin(valid_asins)]
    full_meta.append(filtered)

# Combine into one final DataFrame
df_final = pd.concat(full_meta, ignore_index=True)

print(f"Success! Found {len(df_final)} matching products.")

Reading with built-in JSON parser...
Success! Found 134754 matching products.


In [30]:
df_final.to_csv("Health_and_Household_Final.csv", sep='\x1f', index=False)

In [4]:
df_meta = pd.read_csv("Health_and_Household_Final.csv", sep='\x1f')

C:\Users\Ronza\AppData\Local\Temp\ipykernel_41428\2538684490.py:1: DtypeWarning: Columns (6,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df_meta = pd.read_csv("Health_and_Household_Final.csv", sep='\x1f')


In [8]:
df_meta.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Health & Personal Care,InvoSpa Shiatsu Back Shoulder and Neck Massage...,3.7,15,['8 Massage roller balls - This corded shoulde...,[],49.97,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'InvoSpa Neck and Shoulder Massager...,InvoSpa,"['Health & Household', 'Wellness & Relaxation'...","{'Use for': 'Legs, Shoulder, Neck, Feet, Foot'...",B0C4L5Y711,NaN,NaN,NaN
1,NaN,Zippo Eagle Lighters,4.7,2990,['Genuine Zippo windproof lighter with distinc...,"[""Our eagle lighter collection were created as...",19.31,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Zippo Eagle Lighters ', 'url': 'ht...",Zippo,"['Health & Household', 'Household Supplies', '...",{'Item Package Dimensions L x W x H': '3.35 x ...,B01CYPB41C,NaN,NaN,NaN
2,AMAZON FASHION,DAWGS Women's Loudmouth Patterns Z Sandals | L...,4.5,5161,[],['The LoudMouth Z is DAWGS\' most popular sand...,39.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],DAWGS,"['Health & Household', 'Health Care', 'Foot He...","{'Is Discontinued By Manufacturer': 'No', 'Pac...",B00VMGV2DK,NaN,NaN,NaN
3,Health & Personal Care,Optimum Nutrition Amino Energy Naturally Flavo...,4.1,927,"['NATURALLY FLAVORED – no artificial flavors, ...","[""Now there's a new way to enjoy anytime energ...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Optimum Nutrition Amino Energy, Co...",Optimum Nutrition,"['Health & Household', 'Diet & Sports Nutritio...","{'Brand': 'Optimum Nutrition', 'Item Form': 'P...",B074K5KR9K,NaN,NaN,NaN
4,Health & Personal Care,Caltrate Chewables 600 Plus D3 Plus Minerals C...,4.4,12601,['155 count bottle of Caltrate Chewables 600 P...,['Caltrate Chewables 600 Plus D3 Plus Minerals...,15.72,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'These Taste Great, Motivates to Ta...",Caltrate,"['Health & Household', 'Vitamins, Minerals & S...","{'Brand': 'Caltrate', 'Item Form': 'Chewables'...",B0C4V9Y9VB,NaN,NaN,NaN


In [7]:
sample_df = sample_df.sample(n=500000)

718198